## Import packages

In [1]:
import os
import sys
import json
import argparse
import numpy as np
import math
from einops import rearrange
import time
import random
import string
import h5py
from tqdm import tqdm
import webdataset as wds
import gc

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms

# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True

## Configuration

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
data_type = torch.float16 # change depending on your mixed_precision
num_devices = torch.cuda.device_count()
batch_size = 8
num_epochs=12

print(f"data_type={data_type}, num_devices={num_devices}, batch_size={batch_size}, num_epochs={num_epochs}")

data_path = "/teamspace/studios/this_studio/nsd"
subj = 1
subj_list = [subj]
num_sessions = 2
num_test = 1985
num_voxels_list = []

def my_split_by_node(urls): 
    return urls

print(f"data_path={data_path}, subj={subj}, subj_list={subj_list}, num_sessions={num_sessions}, num_test={num_test}, num_voxels_list={num_voxels_list}")

data_type=torch.float16, num_devices=1, batch_size=8, num_epochs=12
data_path=/teamspace/studios/this_studio/nsd, subj=1, subj_list=[1], num_sessions=2, num_test=1985, num_voxels_list=[]


## Creating wds dataloader

In [3]:
train_data = {}
train_dl = {}
num_voxels = {}
voxels = {}

for s in subj_list:

    train_url = f"{data_path}/wds/subj0{s}/train/" + "{0.." + f"{num_sessions-1}" + "}.tar"

    print(train_url)
    
    train_data[f'subj0{s}'] = wds.WebDataset(train_url,resampled=True,nodesplitter=my_split_by_node)\
                        .shuffle(750, initial=1500, rng=random.Random(42))\
                        .decode("torch")\
                        .rename(behav="behav.npy", past_behav="past_behav.npy", future_behav="future_behav.npy", olds_behav="olds_behav.npy")\
                        .to_tuple(*["behav", "past_behav", "future_behav", "olds_behav"])

    train_dl[f'subj0{s}'] = torch.utils.data.DataLoader(train_data[f'subj0{s}'], batch_size=batch_size, shuffle=False, drop_last=True, pin_memory=True)

    f = h5py.File(f'{data_path}/betas_all_subj0{s}_fp32_renorm.hdf5', 'r')
    betas = f['betas'][:]
    betas = torch.Tensor(betas).to("cpu").to(data_type)
    num_voxels_list.append(betas[0].shape[-1])
    num_voxels[f'subj0{s}'] = betas[0].shape[-1]
    voxels[f'subj0{s}'] = betas

    print(f"num_voxels for subj0{s}: {num_voxels[f'subj0{s}']}")

print("Loaded all subj train dls and betas!\n")

test_url = f"{data_path}/wds/subj0{subj}/test/" + "0.tar"

test_data = wds.WebDataset(test_url,resampled=False,nodesplitter=my_split_by_node)\
                    .shuffle(750, initial=1500, rng=random.Random(42))\
                    .decode("torch")\
                    .rename(behav="behav.npy", past_behav="past_behav.npy", future_behav="future_behav.npy", olds_behav="olds_behav.npy")\
                    .to_tuple(*["behav", "past_behav", "future_behav", "olds_behav"])
test_dl = torch.utils.data.DataLoader(test_data, batch_size=num_test, shuffle=False, drop_last=True, pin_memory=True)

print(f"Loaded test dl for subj{subj}!\n")

/teamspace/studios/this_studio/nsd/wds/subj01/train/{0..1}.tar
num_voxels for subj01: 15724
Loaded all subj train dls and betas!

Loaded test dl for subj1!



In [4]:
f = h5py.File(f'{data_path}/coco_images_224_float16.hdf5', 'r')
images = f['images'][:batch_size] # if you go OOM you can remove the [:] so it isnt preloaded to cpu! (will require a few edits elsewhere though)
images = torch.Tensor(images).to(device).to(data_type)
images.shape

torch.Size([8, 3, 224, 224])

## Load models

### CLIP image embeddings model

In [5]:
import clip
from PIL import Image

clip_embedder, preprocess = clip.load("ViT-B/32", device=device)

In [6]:
print(images[0].unsqueeze(0).shape)
encoded_image = clip_embedder.encode_image(image=images[0].unsqueeze(0))
print(encoded_image.shape)

torch.Size([1, 3, 224, 224])
torch.Size([1, 512])


### Mindeye modules

In [7]:
# Input: Flattened representation of image's clip embedding
# Output: Voxel representation

class MindEyeModule(nn.Module):
    def __init__(self):
        super(MindEyeModule, self).__init__()
    def forward(self, x):
        return x
        
model = MindEyeModule()

class RidgeRegression(torch.nn.Module):

    def __init__(self, input_sizes, out_features): # input_sizes will be (batch_size, 768 or whatever's the flattened representation of the clip embedding) and out_features will be (batch_size, num_voxels)
        super(RidgeRegression, self).__init__()
        self.out_features = out_features
        self.linears = torch.nn.Linear(input_sizes, out_features)
    def forward(self, x):
        out = self.linears(x)
        return out
        
model.ridge = RidgeRegression(input_sizes=batch_size * 512, out_features=batch_size * num_voxels[f'subj0{subj}'])

## Main

In [ ]:
epoch = 0
losses, test_losses, lrs = [], [], []
best_test_loss = 1e9
torch.cuda.empty_cache()